## ⚠️ Known issue fixed here: mask-after-inversion (2026-06-14)

Cell 4 originally computed `max_val - (membrane × cellmask)`, which flips the
masked-out 0 background to white. Local Thickness then measured the whole
**outside-the-cell** area as one giant spurious slit gap, biasing per-cell mean
SD width **upward by ~2–5 px (non-uniformly)**. Cell 4 now re-applies the cell
mask *after* inversion so the exterior stays 0.

**Impact on already-published results:** any dataset processed with the previous
version (including the published sister dataset) carries this upward bias and
would need re-running steps 3→4→5→6; a correction/erratum to reported SD widths
may be warranted. See `KNOWN_ISSUE_mask_after_inversion.md`.


In [1]:
# Cell 1: Libraries
# All imports for the notebook - also duplicated in each cell for standalone execution

import os
from pathlib import Path
import numpy as np
from skimage import io
import tifffile

In [2]:
# Cell 2: Paths
import os
from pathlib import Path

# Base/parental path - modify this to your data location
BASE_PATH = Path("/Users/pavel/Downloads/")

In [9]:
# Cell 3: Invert masks and padding, subtract and save
import os
from pathlib import Path
import numpy as np
from skimage import io
import tifffile

# Define subfolders
masks_folder = BASE_PATH / "003_unet_segmentation" / "masks"
padding_folder = BASE_PATH / "002_padding"
output_folder = BASE_PATH / "004_local_thickness_input"

# Create output folder if it doesn't exist
output_folder.mkdir(parents=True, exist_ok=True)

# Get all TIFF files from masks folder (ignore non-tif and meta files)
mask_files = [f for f in masks_folder.iterdir() 
              if f.suffix.lower() in ['.tif', '.tiff'] 
              and '_mask' in f.stem
              and not f.name.startswith('.')]

print(f"Found {len(mask_files)} mask files")

for mask_file in mask_files:
    # Extract base name (remove _mask suffix)
    base_name = mask_file.stem.replace('_mask', '')
    
    # Find corresponding padding file
    padding_file = None
    for ext in ['.tif', '.tiff']:
        potential_padding = padding_folder / f"{base_name}_padding{ext}"
        if potential_padding.exists():
            padding_file = potential_padding
            break
    
    if padding_file is None:
        print(f"Warning: No padding file found for {mask_file.name}, skipping...")
        continue
    
    # Read images
    mask_img = tifffile.imread(mask_file)
    padding_img = tifffile.imread(padding_file)
    
    print(f"Processing: {base_name}")
    print(f"  Mask shape: {mask_img.shape}, dtype: {mask_img.dtype}")
    print(f"  Padding shape: {padding_img.shape}, dtype: {padding_img.dtype}")
    
    # Determine max value for inversion based on dtype
    if mask_img.dtype == np.uint8:
        max_val = 255
    elif mask_img.dtype == np.uint16:
        max_val = 65535
    else:
        max_val = mask_img.max()
    
    # Normalize padding to [0, 1]
   # mask_inverted = mask_img # max_val - mask_img
    padding_normalized = padding_img / 255 #max_val - padding_img
    
    # Subtract inverted padding from inverted mask
    # Use clip to avoid negative values
    result = np.clip(mask_img.astype(np.int32) * padding_normalized.astype(np.int32), 0, max_val).astype(mask_img.dtype)
    
    # Save result
    output_file = output_folder / f"{base_name}.tif"
    tifffile.imwrite(output_file, result)
    print(f"  Saved: {output_file.name}")

print(f"\nDone! Processed {len(mask_files)} files.")

Found 4 mask files
Processing: A15_ROI000
  Mask shape: (2, 466, 518), dtype: uint8
  Padding shape: (466, 518), dtype: uint8
  Saved: A15_ROI000.tif
Processing: A15_ROI001
  Mask shape: (486, 534), dtype: uint8
  Padding shape: (486, 534), dtype: uint8
  Saved: A15_ROI001.tif
Processing: E29-ROI-000
  Mask shape: (2, 466, 518), dtype: uint8
  Padding shape: (466, 518), dtype: uint8
  Saved: E29-ROI-000.tif
Processing: E29-ROI-001
  Mask shape: (486, 534), dtype: uint8
  Padding shape: (486, 534), dtype: uint8
  Saved: E29-ROI-001.tif

Done! Processed 4 files.


In [ ]:
# Cell 4: Split channels into membranes and clusters
import os
from pathlib import Path
import numpy as np
import tifffile

# Define folders
input_folder = BASE_PATH / "004_local_thickness_input"
membranes_folder = BASE_PATH / "005_membranes"
clusters_folder = BASE_PATH / "006_clusters"

# Create output folders if they don't exist
membranes_folder.mkdir(parents=True, exist_ok=True)
clusters_folder.mkdir(parents=True, exist_ok=True)

# Get all TIFF files from input folder
input_files = [f for f in input_folder.iterdir() 
               if f.suffix.lower() in ['.tif', '.tiff'] 
               and not f.name.startswith('.')]

print(f"Found {len(input_files)} files to split")

for input_file in input_files:
    base_name = input_file.stem
    
    # Read image
    img = tifffile.imread(input_file)
    
    print(f"Processing: {base_name}")
    print(f"  Shape: {img.shape}, dtype: {img.dtype}")
    
    max_val = np.iinfo(img.dtype).max

    # Check if image has channels (3D array with first dim = channels)
    if img.ndim == 3 and img.shape[0] == 2:
        # Split channels
        channel_1 = img[0]  # First channel - membranes
        channel_2 = img[1]  # Second channel - clusters
    elif img.ndim == 2:
        # Single channel image - use same for both (or skip)
        print(f"  Warning: Single channel image, using same data for both outputs")
        channel_1 = img
        channel_2 = img
    else:
        print(f"  Warning: Unexpected shape {img.shape}, skipping...")
        continue
    
    # Invert membrane channel: membranes become 0 (black), gaps become max_val (white)
    # Local Thickness measures white structures, so this makes it measure gap widths
    channel_1 = max_val - channel_1

    # FIX (mask-after-inversion, 2026-06-14): re-zero everything outside the cell.
    # Without this, the inversion above turns the masked-out 0 background into
    # max_val (white), and Local Thickness measures the whole outside-cell region as
    # one giant spurious gap (mean ~528 px, up to 747 px). Most is >100 px and dropped
    # by the step-5 0-100 px bins, but a <100 px rim leaks in and the white sea floods
    # perimeter gaps inside the mask, biasing per-cell mean SD width upward ~2-5 px.
    padding_folder = BASE_PATH / "002_padding"
    padding_file = None
    for ext in ['.tif', '.tiff']:
        cand = padding_folder / f"{base_name}_padding{ext}"
        if cand.exists():
            padding_file = cand
            break
    if padding_file is None:
        print(f"  Warning: no padding/cell-mask for {base_name}; skipping re-mask")
    else:
        cell_mask = tifffile.imread(padding_file) > 0
        channel_1 = np.where(cell_mask, channel_1, 0).astype(img.dtype)

    # Save membrane channel
    membrane_file = membranes_folder / f"{base_name}_membrane.tif"
    tifffile.imwrite(membrane_file, channel_1)
    print(f"  Saved membrane: {membrane_file.name}")
    
    # Save clusters channel
    clusters_file = clusters_folder / f"{base_name}_clusters.tif"
    tifffile.imwrite(clusters_file, channel_2)
    print(f"  Saved clusters: {clusters_file.name}")

print(f"\nDone! Split {len(input_files)} files into membranes and clusters.")